# Interactive Resonance Matcher
Match resonances between two VNA datasets.

## Quick Start Guide

### What This Tool Does
The resonance matcher helps you pair resonances between two VNA datasets. Each resonance gets a unique `res_idx`. The tool creates **match groups** linking resonances between datasets, and allows the user to remove or insert resonators into datasets during the process.

### Basic Workflow

1. **Run the matcher** (see cell below) - opens an interactive window
2. **Review automatic matches** - colored markers show groups (same color = same group)
3. **Multi-select and adjust** using Ctrl+click and keyboard shortcuts
4. **Close the window** - automatically saves to zarr

The window has:
- **Three plot panels:** Overview navigator (top), Magnitude (middle), Phase (bottom)
- **Right sidebar:** Toolbar with quick actions + controls
- **Yellow dashed line:** Re-match threshold

**Visual encoding:**
- Filled circles = Dataset 1 (DS1)
- Open squares = Dataset 2 (DS2)
- Same color = same match group
- Dashed border = flagged as ambiguous (auto-set for multi-res groups)
- White ring(s) = selected resonance(s) — **auto-deselected when scrolled off screen**

### Multi-Selection 

**To select resonances:**
1. **Left-click** on a resonance → selects it (white ring), replaces previous selection
2. **Ctrl+Left-click** on another resonance → adds to selection (multiple white rings)
3. Repeat to select across DS1 and DS2
4. Status bar shows: "Selected: 5 resonances (DS1: 3, DS2: 2) in 2 groups"
5. **Pan away** → out-of-view resonances auto-deselect

### Navigation (Left Hand Keyboard)

**Pan/Zoom:**
- Mouse wheel: zoom in/out
- Drag the yellow region in the overview panel
- `Z`/`X`: pan left/right 20% (fine adjustment)
- `A`/`S`: pan left/right 80% (fast jumps)

**Pro tip:** Keys arranged for left hand - Z/X for fine control, A/S for fast jumps

### Merging Resonances (Two Modes!)

**F key: Merge Groups**
- Merges ALL groups containing selected resonances
- Keeps all resonances from those groups together
- Example: groups `((1,2,3),(1))` and `((4),(2))`
  - Select DS1: 2,4 and DS2: 2
  - Press `F` → `((1,2,3,4),(1,2))`

**D key: Merge Selected Only**
- Creates new group with ONLY selected resonances
- Unlinks others from their original groups
- Same example:
  - Select DS1: 2,4 and DS2: 2
  - Press `D` → `((2,4),(2))` and `((1,3),(1))`

**Use cases:**
- `F` when you want to combine groups completely
- `D` when you want to pull specific resonances together and separate their original groups

### Other Editing

**E key: Delete Selected**
- Deletes all selected resonances

**W key: Unlink Selected**
- Puts each selected resonance in its own separate group
- Useful for splitting apart incorrectly matched groups

**Q key: Toggle Ambiguous**
- Only works on single 1:1 matches
- Groups with multiple resonances in either dataset are auto-flagged ambiguous
- Use when you're uncertain about a 1:1 pairing

### Adding Resonances

**Shift+Left-click on empty space:**
- Shift+click anywhere on the magnitude or phase plot 
- If previously deleted resonances exist nearby, dialog asks: reuse old `res_idx` or create new?

### Re-Match Threshold (Yellow Line)

**Three modes:**

1. **Pinned to right edge (on startup)**
   - Line starts at right edge of window
   - **Stays at right edge as you pan** → always visible, ready for re-matching

2. **Fixed at frequency (after edits)**
   - After delete/add/merge/unlink, line moves to right of your edit
   - **Stays at that frequency as you pan** → marks boundary between corrected and uncorrected
   - Example: Fix resonances at 4-5 GHz, line sits at ~5 GHz, pan to 6-7 GHz (line off-screen)

3. **Snap back to right (when scrolled out)**
   - If you pan fully right of the fixed-frequency line, it snaps back to right edge
   - Returns to pinned mode → ready for more work

**To re-match everything above the line:**
1. Press `R` (or **Re-Match** button)
2. All groups above line are dissolved and re-matched automatically
3. Your manual corrections below line stay unchanged

**Manual override:** Ctrl+right-click to set threshold at any frequency (stays fixed there)

### Ambiguous Flag

- **Automatically True** for groups with multiple resonances in either dataset
  - (2,3) ↔ (1) → ambiguous
  - (1) ↔ (2,3) → ambiguous
  - (2) ↔ (2,3) → ambiguous
- **Q key only toggles** for 1:1 matches where you're uncertain
  - (1) ↔ (1) → Q to mark as ambiguous if you're unsure about the pairing

### Keyboard Reference

**Navigation:**
- `Z` / `X` - pan 20% left/right
- `A` / `S` - pan 80% left/right
- Scroll wheel - zoom

**Editing:**
- `E` - delete selected resonances
- `F` - merge groups (all groups with selected resonances)
- `D` - merge selected only (create new group from selection)
- `W` - unlink selected (each gets own group)
- `Q` - toggle ambiguous flag (1:1 matches only)
- `R` - re-match above threshold

**Other:**
- `H` - show help dialog
- `Ctrl+S` - save to zarr
- `Ctrl+Z` - undo (max 50 steps)

### Saving and Loading

**Auto-saves on close** - just close the window (X button) and data is written to zarr.

Or use:
- `Ctrl+S`: save now (keeps window open)
- Toolbar **Save** button

**Loading from zarr:**
When you run `run_res_matcher()` and zarr data already exists, you'll see a dialog with three options:

1. **Overwrite** - Delete existing data and start fresh with automatic matching (asks for confirmation)
2. **Load from Zarr** - Load your existing groups and continue editing where you left off
3. **Cancel** - Exit without doing anything

**Smart position tracking:**
- The tool tracks the furthest right position you've viewed (left edge of window)
- When loading from zarr, it centers the window on this saved position
- Makes it easy to continue your work from where you left off
- Saved in zarr as `max_view_left`

In [ ]:
import zarr
import numpy as np
from citkid.vna.res_matcher import run_res_matcher

# Load your data - replace these with your actual data loading code
# f1, f2: frequency arrays in Hz
# z1, z2: complex S21 arrays  
# fres1, fres2: 1D arrays of resonance frequencies (Hz) found by peak finder
# res_idx1, res_idx2: integer resonator indices (must be unique within each dataset)

f1, z1, fres1, res_idx1 = None, None, None, None
f2, z2, fres2, res_idx2 = None, None, None, None

# Open a Zarr group to save the matched output
output_path = 'path/to/output.zarr'  # Change this
zarr_grp = zarr.open(output_path, mode='a')

In [ ]:
# Run the interactive matcher

groups = run_res_matcher(
    f1, z1, fres1, res_idx1,  # Dataset 1
    f2, z2, fres2, res_idx2,  # Dataset 2
    zarr_grp=zarr_grp,         # Where to save output
    new_res_start_idx=2000,    # New resonances get indices starting here
    init_match='sorted',       # 'sorted' or 'nearest' for initial pairing
    apply_filter=False         # Set True to start with highpass filter on
)

# Output arrays are now in zarr_grp:
# grp_ids are indices that keep track of which group each resonance belongs to
# - fres1_out, res_idx1_out, group_ids1  (DS1 resonances)
# - fres2_out, res_idx2_out, group_ids2  (DS2 resonances)
# - ambiguous_groups (list of flagged group IDs)
# - max_view_left (highest left edge position you've viewed)